In [1]:
import json
import gc
import numpy as np
from types import SimpleNamespace
from rouge_score import rouge_scorer as rouge_scoring

from verbatim_rag.extractors import LLMSpanExtractor, SemanticHighlightExtractor
from verbatim_rag.core import VerbatimRAG

In [2]:
import json

def load_jsonl(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

dev_answerable = load_jsonl('../../data/clapnq_dev_answerable.jsonl')
dev_unanswerable = load_jsonl('../../data/clapnq_dev_unanswerable.jsonl')

print(f"Answerable: {len(dev_answerable)}")
print(f"Unanswerable: {len(dev_unanswerable)}")

Answerable: 300
Unanswerable: 300


### helper functions

In [3]:
def is_unanswerable_response(answer):
    """Check if the answer indicates 'I don't know'."""
    if not answer:
        return True
    unanswerable_phrases = [
        "i don't know", "i do not know", "cannot be answered",
        "not answerable", "no answer", "unanswerable",
        "cannot answer", "not enough information", "insufficient information",
    ]
    answer_lower = answer.lower().strip()
    return any(phrase in answer_lower for phrase in unanswerable_phrases)


def compute_rougeLp(generated_answer, passage_text):
    """RougeLp: Faithfulness to the passage."""
    scorer = rouge_scoring.RougeScorer(['rougeL'], use_stemmer=False)
    scores = scorer.score(passage_text, generated_answer)
    return scores['rougeL'].precision

### run functions

In [4]:
def run_single_llm(example):
    """VerbatimRAG with LLMSpanExtractor."""
    question = example['input']
    passage_text = example['passages'][0]['text']

    search_results = [SimpleNamespace(text=passage_text)]
    extractor = LLMSpanExtractor(span_match_mode="fuzzy")
    relevant_spans = extractor.extract_spans(question, search_results)

    if not relevant_spans or all(len(v) == 0 for v in relevant_spans.values()):
        return ""

    verbatim_rag = VerbatimRAG(index=None)
    display_spans, citation_spans = verbatim_rag._rank_and_split_spans(relevant_spans)
    system_answer = verbatim_rag.template_manager.process(
        question, display_spans, citation_spans
    )
    return system_answer


def run_single_semantic(example):
    """VerbatimRAG with SemanticHighlightExtractor."""
    question = example['input']
    passage_text = example['passages'][0]['text']

    search_results = [SimpleNamespace(text=passage_text)]
    relevant_spans = semantic_extractor.extract_spans(question, search_results)

    if not relevant_spans or all(len(v) == 0 for v in relevant_spans.values()):
        return ""

    verbatim_rag = VerbatimRAG(index=None)
    display_spans, citation_spans = verbatim_rag._rank_and_split_spans(relevant_spans)
    system_answer = verbatim_rag.template_manager.process(
        question, display_spans, citation_spans
    )
    return system_answer

from verbatim_core.templates import TemplateManager

def run_single_llm_spans_only(example):
    """VerbatimRAG with LLMSpanExtractor, spans only (no template wrapper)."""
    question = example['input']
    passage_text = example['passages'][0]['text']

    search_results = [SimpleNamespace(text=passage_text)]
    extractor = LLMSpanExtractor(span_match_mode="fuzzy")
    relevant_spans = extractor.extract_spans(question, search_results)

    if not relevant_spans or all(len(v) == 0 for v in relevant_spans.values()):
        return ""

    tm = TemplateManager(default_mode="static")
    tm.use_static_mode(template="[DISPLAY_SPANS]")
    tm.set_citation_mode("hidden") 
    verbatim_rag = VerbatimRAG(index=None, template_manager=tm)
    display_spans, citation_spans = verbatim_rag._rank_and_split_spans(relevant_spans)
    return verbatim_rag.template_manager.process(question, display_spans, citation_spans)


def run_single_semantic_spans_only(example):
    """VerbatimRAG with SemanticHighlightExtractor, spans only (no template wrapper)."""
    question = example['input']
    passage_text = example['passages'][0]['text']

    search_results = [SimpleNamespace(text=passage_text)]
    relevant_spans = semantic_extractor.extract_spans(question, search_results)

    if not relevant_spans or all(len(v) == 0 for v in relevant_spans.values()):
        return ""

    tm = TemplateManager(default_mode="static")
    tm.use_static_mode(template="[DISPLAY_SPANS]")
    tm.set_citation_mode("hidden") 
    verbatim_rag = VerbatimRAG(index=None, template_manager=tm)
    display_spans, citation_spans = verbatim_rag._rank_and_split_spans(relevant_spans)
    return verbatim_rag.template_manager.process(question, display_spans, citation_spans)

### eval functions

In [5]:
def evaluate_generation(
    dev_answerable, 
    dev_unanswerable, 
    n_samples=None
):
    """Original evaluation function for LLMSpanExtractor."""
    if n_samples:
        answerable_data = dev_answerable[:n_samples]
        unanswerable_data = dev_unanswerable[:n_samples]
    else:
        answerable_data = dev_answerable
        unanswerable_data = dev_unanswerable
    
    scorer = rouge_scoring.RougeScorer(['rougeL', 'rouge1'], use_stemmer=False)
    
    # PART 1: Answerable Questions
    print(f"\nEvaluating {len(answerable_data)} ANSWERABLE examples...")
    
    rougeL_list = []
    recall_list = []
    rougeLp_list = []
    length_list = []
    
    for i, example in enumerate(answerable_data):
        question = example['input']
        passage_text = example['passages'][0]['text']
        reference_answer = example['output'][0]['answer']
        
        try:
            system_answer = run_single_llm(example)
        except Exception as e:
            print(f"Error on example {i+1}: {e}")
            system_answer = ""
        
        if system_answer:
            scores = scorer.score(reference_answer, system_answer)
            rougeL_list.append(scores['rougeL'].fmeasure * 100)
            recall_list.append(scores['rouge1'].recall * 100)
            rougeLp_list.append(compute_rougeLp(system_answer, passage_text) * 100)
            length_list.append(len(system_answer))
        else:
            rougeL_list.append(0.0)
            recall_list.append(0.0)
            rougeLp_list.append(0.0)
            length_list.append(0)
        
        if (i + 1) % 10 == 0:
            print(f"  Progress: {i+1}/{len(answerable_data)}")
    
    # PART 2: Unanswerable Questions
    print(f"\nEvaluating {len(unanswerable_data)} UNANSWERABLE examples...")
    
    unanswerable_correct = []
    
    for i, example in enumerate(unanswerable_data):
        try:
            system_answer = run_single_llm(example)
        except Exception as e:
            print(f"Error on example {i+1}: {e}")
            system_answer = ""
        
        is_correct = is_unanswerable_response(system_answer)
        unanswerable_correct.append(1.0 if is_correct else 0.0)
        
        if (i + 1) % 10 == 0:
            print(f"  Progress: {i+1}/{len(unanswerable_data)}")
    
    # RESULTS
    print(f"\n{'='*60}")
    print(f"{'='*60}")
    print(f"\nANSWERABLE (n={len(answerable_data)}):")
    print(f"  RougeL:     {np.mean(rougeL_list):.1f}")
    print(f"  Recall:     {np.mean(recall_list):.1f}")
    print(f"  RougeLp:    {np.mean(rougeLp_list):.1f}")
    print(f"  Avg Length: {np.mean(length_list):.0f} chars")
    print(f"\nUNANSWERABLE (n={len(unanswerable_data)}):")
    print(f"  Unanswerable Accuracy: {np.mean(unanswerable_correct)*100:.1f}%")
    print(f"\n{'='*60}")
    print(f"PAPER BASELINES (Dev):")
    print(f"{'='*60}")
    print(f"  GPT-3.5:      RougeL=39.8 Recall=58.9 RougeLp=30.0 Unans=37.0%")
    print(f"  CLAPNQ-T5-LG: RougeL=57.2 Recall=68.3 RougeLp=51.0 Unans=89.2%")
    print(f"{'='*60}")

    # am Ende der Funktion, vor dem letzten print
    return {
        'rougeL': float(np.mean(rougeL_list)),
        'recall': float(np.mean(recall_list)),
        'rougeLp': float(np.mean(rougeLp_list)),
        'avg_len': float(np.mean(length_list)),
        'unans_acc': float(np.mean(unanswerable_correct) * 100),
    }

In [6]:
def evaluate_in_batches(data, run_fn, batch_size=50, data_type="answerable", label="system"):
    """Evaluate in batches with memory cleanup and intermediate saves."""
    scorer = rouge_scoring.RougeScorer(['rougeL', 'rouge1'], use_stemmer=False)

    all_rougeL, all_recall, all_rougeLp, all_length, all_unans = [], [], [], [], []

    for batch_start in range(0, len(data), batch_size):
        batch_end = min(batch_start + batch_size, len(data))
        batch = data[batch_start:batch_end]
        print(f"\n--- Batch {batch_start}-{batch_end} / {len(data)} ({data_type}) ---")

        for i, example in enumerate(batch):
            try:
                system_answer = run_fn(example)
            except Exception as e:
                print(f"Error on example {batch_start + i}: {e}")
                system_answer = ""

            if data_type == "answerable":
                passage_text = example['passages'][0]['text']
                reference_answer = example['output'][0]['answer']
                if system_answer:
                    scores = scorer.score(reference_answer, system_answer)
                    all_rougeL.append(scores['rougeL'].fmeasure * 100)
                    all_recall.append(scores['rouge1'].recall * 100)
                    all_rougeLp.append(compute_rougeLp(system_answer, passage_text) * 100)
                    all_length.append(len(system_answer))
                else:
                    all_rougeL.append(0.0)
                    all_recall.append(0.0)
                    all_rougeLp.append(0.0)
                    all_length.append(0)
            else:
                all_unans.append(1.0 if is_unanswerable_response(system_answer) else 0.0)

        # Memory cleanup after each batch
        gc.collect()

        # Save intermediate results
        if data_type == "answerable":
            interim = {
                'rougeL': float(np.mean(all_rougeL)),
                'recall': float(np.mean(all_recall)),
                'rougeLp': float(np.mean(all_rougeLp)),
                'avg_len': float(np.mean(all_length)),
                'n_done': batch_end,
            }
        else:
            interim = {'unans_acc': float(np.mean(all_unans) * 100), 'n_done': batch_end}

        with open(f'{label}_{data_type}_interim.json', 'w') as f:
            json.dump(interim, f, indent=2)
        print(f"  Saved. ({batch_end}/{len(data)})")

    if data_type == "answerable":
        return {
            'rougeL': float(np.mean(all_rougeL)),
            'recall': float(np.mean(all_recall)),
            'rougeLp': float(np.mean(all_rougeLp)),
            'avg_len': float(np.mean(all_length)),
        }
    else:
        return {'unans_acc': float(np.mean(all_unans) * 100)}

## LLMSpanExtractor

In [8]:

results_llm = evaluate_generation(
    dev_answerable, 
    dev_unanswerable, 
    n_samples=None
)


Evaluating 300 ANSWERABLE examples...
  Progress: 10/300


  Progress: 20/300
  Progress: 30/300
  Progress: 40/300
  Progress: 50/300
  Progress: 60/300
  Progress: 70/300
  Progress: 80/300
  Progress: 90/300
  Progress: 100/300
  Progress: 110/300


  Progress: 120/300
  Progress: 130/300
  Progress: 140/300
  Progress: 150/300
  Progress: 160/300
  Progress: 170/300
  Progress: 180/300
  Progress: 190/300
  Progress: 200/300
  Progress: 210/300
  Progress: 220/300
  Progress: 230/300
  Progress: 240/300
  Progress: 250/300


  Progress: 260/300


  Progress: 270/300
  Progress: 280/300


  Progress: 290/300
  Progress: 300/300

Evaluating 300 UNANSWERABLE examples...
  Progress: 10/300
  Progress: 20/300
  Progress: 30/300
  Progress: 40/300
  Progress: 50/300
  Progress: 60/300
  Progress: 70/300
  Progress: 80/300
  Progress: 90/300
  Progress: 100/300
  Progress: 110/300
  Progress: 120/300
  Progress: 130/300
  Progress: 140/300
  Progress: 150/300
  Progress: 160/300
  Progress: 170/300
  Progress: 180/300
  Progress: 190/300
  Progress: 200/300
  Progress: 210/300
  Progress: 220/300
  Progress: 230/300
  Progress: 240/300
  Progress: 250/300
  Progress: 260/300
  Progress: 270/300
  Progress: 280/300


  Progress: 290/300
  Progress: 300/300


ANSWERABLE (n=300):
  RougeL:     47.4
  Recall:     73.8
  RougeLp:    64.3
  Avg Length: 481 chars

UNANSWERABLE (n=300):
  Unanswerable Accuracy: 77.0%

PAPER BASELINES (Dev):
  GPT-3.5:      RougeL=39.8 Recall=58.9 RougeLp=30.0 Unans=37.0%
  CLAPNQ-T5-LG: RougeL=57.2 Recall=68.3 RougeLp=51.0 Unans=89.2%


In [11]:
with open('llm_results.json', 'w') as f:
    json.dump(results_llm, f, indent=2)

## semanticHighlightExtractor

bevor wir es ausführen müssen wir kurz testen welche inputs für das mdoell am besten sind

In [ ]:
# configs = [
#     {"threshold": 0.3, "output_mode": "sentences"},
#     {"threshold": 0.5, "output_mode": "sentences"},
#     {"threshold": 0.7, "output_mode": "sentences"},
#     {"threshold": 0.3, "output_mode": "spans"},
#     {"threshold": 0.5, "output_mode": "spans"},
#     {"threshold": 0.7, "output_mode": "spans"},
# ]

# sweep_results = []

# for cfg in configs:
#     print(f"\n--- threshold={cfg['threshold']}, mode={cfg['output_mode']} ---")
    
#     semantic_extractor = SemanticHighlightExtractor(
#         threshold=cfg['threshold'],
#         output_mode=cfg['output_mode'],
#     )
    
#     ans = evaluate_in_batches(
#         dev_answerable[:20], run_single_semantic,
#         batch_size=20, data_type="answerable",
#         label=f"sweep_t{cfg['threshold']}_{cfg['output_mode']}"
#     )
    
#     unans = evaluate_in_batches(
#         dev_unanswerable[:20], run_single_semantic,
#         batch_size=20, data_type="unanswerable",
#         label=f"sweep_t{cfg['threshold']}_{cfg['output_mode']}"
#     )
    
#     sweep_results.append({
#         'threshold': cfg['threshold'],
#         'output_mode': cfg['output_mode'],
#         **ans,
#         **unans,
#     })
    
#     del semantic_extractor
#     gc.collect()

# # Save sweep results
# with open('sweep_results.json', 'w') as f:
#     json.dump(sweep_results, f, indent=2)

# # Display table
# print(f"\n{'='*80}")
# print(f"THRESHOLD / OUTPUT MODE SWEEP (n=20)")
# print(f"{'='*80}")
# print(f"{'Threshold':<12} {'Mode':<12} {'RougeL':>7} {'Recall':>7} {'RougeLp':>8} {'Len':>5} {'Unans':>7}")
# print(f"{'-'*80}")
# for r in sweep_results:
#     print(f"{r['threshold']:<12} {r['output_mode']:<12} {r['rougeL']:>7.1f} {r['recall']:>7.1f} {r['rougeLp']:>8.1f} {r['avg_len']:>5.0f} {r['unans_acc']:>6.1f}%")
# print(f"{'='*80}")

In [ ]:
# from tabulate import tabulate

# sweep_rows = [
#     (
#         f"{r['threshold']}",
#         r['output_mode'],
#         r['rougeL'],
#         r['recall'],
#         r['rougeLp'],
#         r['avg_len'],
#         r['unans_acc'],
#     )
#     for r in sweep_results
# ]

# headers = ["Threshold", "Mode", "RougeL", "Recall", "RougeLp", "Len", "Unans\\%"]

# print("--- LaTeX ---")
# print(tabulate(sweep_rows, headers=headers, tablefmt="latex_booktabs", floatfmt=".1f"))

# print("\n--- Markdown ---")
# print(tabulate(sweep_rows, headers=headers, tablefmt="pipe", floatfmt=".1f"))

ich bleibe beim defaultwert 0,5 und sentences weil 0,5 die beste balance hat zwischen metrics und unanswerable acc und zwischen snet udn spans ist kein signifikanter unterschied

generell ist zu argumentieren dass wir hier nicht optimieren wollen sondern eigentlic hnur mal schauen wie es grundsätzlich performt also is tdenbke ich mit default zu rechnen ganz ok

In [ ]:
# Load model (only once)
semantic_extractor = SemanticHighlightExtractor(
    threshold=0.5,
    output_mode="sentences",
)

In [ ]:
# 7a. Answerable
ans_results = evaluate_in_batches(
    dev_answerable, run_single_semantic,
    batch_size=50, data_type="answerable", label="semantic"
)
print(ans_results)


--- Batch 0-50 / 300 (answerable) ---


Prepare contexts: 100%|██████████| 1/1 [00:00<00:00, 50.97it/s]


[OpenProvenceModel] Model inference time: 1.91s (1 blocks)


Prepare contexts: 100%|██████████| 1/1 [00:00<00:00, 518.52it/s]


[OpenProvenceModel] Model inference time: 0.86s (1 blocks)


...

[OpenProvenceModel] Model inference time: 0.51s (1 blocks)


Prepare contexts: 100%|██████████| 1/1 [00:00<00:00, 1455.85it/s]


[OpenProvenceModel] Model inference time: 0.47s (1 blocks)


Prepare contexts: 100%|██████████| 1/1 [00:00<00:00, 1479.47it/s]


[OpenProvenceModel] Model inference time: 0.47s (1 blocks)
  Saved. (300/300)
{'rougeL': 36.88128999929759, 'recall': 51.467519185985246, 'rougeLp': 50.491142494069415, 'avg_len': 300.36333333333334}


In [ ]:
# 7b. Unanswerable
unans_results = evaluate_in_batches(
    dev_unanswerable, run_single_semantic,
    batch_size=50, data_type="unanswerable", label="semantic"
)
print(unans_results)


--- Batch 0-50 / 300 (unanswerable) ---


Prepare contexts: 100%|██████████| 1/1 [00:00<00:00, 211.66it/s]

[OpenProvenceModel] Model inference time: 0.81s (1 blocks)


Prepare contexts: 100%|██████████| 1/1 [00:00<00:00, 1065.63it/s]


[OpenProvenceModel] Model inference time: 0.46s (1 blocks)


...

[OpenProvenceModel] Model inference time: 0.82s (1 blocks)


Prepare contexts: 100%|██████████| 1/1 [00:00<00:00, 112.29it/s]


[OpenProvenceModel] Model inference time: 1.14s (1 blocks)


Prepare contexts: 100%|██████████| 1/1 [00:00<00:00, 80.19it/s]


[OpenProvenceModel] Model inference time: 1.22s (1 blocks)
  Saved. (300/300)
{'unans_acc': 91.33333333333333}


In [ ]:
# 7c. Combine and save
results_semantic = {**ans_results, **unans_results}

with open('semantic_results_final.json', 'w') as f:
    json.dump(results_semantic, f, indent=2)

print("SemanticHighlight Results (n=300):")
for k, v in results_semantic.items():
    print(f"  {k}: {v}")

SemanticHighlight Results (n=300):
  rougeL: 36.88128999929759
  recall: 51.467519185985246
  rougeLp: 50.491142494069415
  avg_len: 300.36333333333334
  unans_acc: 91.33333333333333


In [9]:
# LLM Extractor – Spans Only
ans_results_llm_so = evaluate_in_batches(
    dev_answerable, run_single_llm_spans_only,
    batch_size=50, data_type="answerable", label="llm_spans_only"
)
unans_results_llm_so = evaluate_in_batches(
    dev_unanswerable, run_single_llm_spans_only,
    batch_size=50, data_type="unanswerable", label="llm_spans_only"
)
results_llm_spans_only = {**ans_results_llm_so, **unans_results_llm_so}

with open('llm_spans_only_results.json', 'w') as f:
    json.dump(results_llm_spans_only, f, indent=2)
print("LLM Spans Only:", results_llm_spans_only)


--- Batch 0-50 / 300 (answerable) ---


  Saved. (50/300)

--- Batch 50-100 / 300 (answerable) ---


  Saved. (100/300)

--- Batch 100-150 / 300 (answerable) ---


  Saved. (150/300)

--- Batch 150-200 / 300 (answerable) ---
  Saved. (200/300)

--- Batch 200-250 / 300 (answerable) ---
  Saved. (250/300)

--- Batch 250-300 / 300 (answerable) ---


  Saved. (300/300)

--- Batch 0-50 / 300 (unanswerable) ---
  Saved. (50/300)

--- Batch 50-100 / 300 (unanswerable) ---
  Saved. (100/300)

--- Batch 100-150 / 300 (unanswerable) ---
  Saved. (150/300)

--- Batch 150-200 / 300 (unanswerable) ---
  Saved. (200/300)

--- Batch 200-250 / 300 (unanswerable) ---
  Saved. (250/300)

--- Batch 250-300 / 300 (unanswerable) ---


  Saved. (300/300)
LLM Spans Only: {'rougeL': 58.817575084494216, 'recall': 70.07123920729721, 'rougeLp': 96.0402243740121, 'avg_len': 327.86, 'unans_acc': 77.0}


In [ ]:
# Semantic Extractor – Spans Only
ans_results_sem_so = evaluate_in_batches(
    dev_answerable, run_single_semantic_spans_only,
    batch_size=50, data_type="answerable", label="semantic_spans_only"
)
unans_results_sem_so = evaluate_in_batches(
    dev_unanswerable, run_single_semantic_spans_only,
    batch_size=50, data_type="unanswerable", label="semantic_spans_only"
)
results_semantic_spans_only = {**ans_results_sem_so, **unans_results_sem_so}

with open('semantic_spans_only_results.json', 'w') as f:
    json.dump(results_semantic_spans_only, f, indent=2)
print("Semantic Spans Only:", results_semantic_spans_only)


--- Batch 0-50 / 300 (answerable) ---


Prepare contexts: 100%|██████████| 1/1 [00:00<00:00, 253.25it/s]


[OpenProvenceModel] Model inference time: 0.80s (1 blocks)


Prepare contexts: 100%|██████████| 1/1 [00:00<00:00, 241.37it/s]


[OpenProvenceModel] Model inference time: 0.85s (1 blocks)


...

[OpenProvenceModel] Model inference time: 0.86s (1 blocks)


Prepare contexts: 100%|██████████| 1/1 [00:00<00:00, 78.14it/s]


[OpenProvenceModel] Model inference time: 1.16s (1 blocks)


Prepare contexts: 100%|██████████| 1/1 [00:00<00:00, 65.98it/s]


[OpenProvenceModel] Model inference time: 1.27s (1 blocks)
  Saved. (300/300)
Semantic Spans Only: {'rougeL': 44.14655738156403, 'recall': 46.00132471450483, 'rougeLp': 77.66666666666667, 'avg_len': 192.72333333333333, 'unans_acc': 91.33333333333333}


## Final Comparison

In [12]:
import json
from tabulate import tabulate

# Ergebnisse laden
with open('llm_results.json') as f:
    results_llm = json.load(f)
with open('llm_spans_only_results.json') as f:
    results_llm_spans_only = json.load(f)
with open('semantic_results_final.json') as f:
    results_semantic = json.load(f)
with open('semantic_spans_only_results.json') as f:
    results_semantic_spans_only = json.load(f)


paper_baselines = [
    ("FLAN-T5-XXL 1/0",    31.9, 23.6, 15.0,  75, 78.1),
    ("Llama-13B-chat",      35.5, 64.3, 34.0, 491, 25.0),
    ("Mistral-7B-Instruct", 39.0, 56.0, 29.0, 384, 18.6),
    ("GPT-3.5",             39.8, 58.9, 30.0, 444, 37.0),
    ("GPT-4",               35.9, 67.7, 30.0, 759, 18.0),
    ("CLAPNQ-T5-LG-200",    41.5, 51.3, 42.1, 272, 89.7),
    ("CLAPNQ-T5-LG",        57.2, 68.3, 51.0, 318, 89.2),
    ("Full Passage",        49.5, 97.4,100.0, 912,  0.0),
]

our_results = [
    ("VerbatimRAG-LLM (contextual template)",
        results_llm['rougeL'], results_llm['recall'],
        results_llm['rougeLp'], results_llm['avg_len'], results_llm['unans_acc']),
    ("VerbatimRAG-LLM (spans only)",
        results_llm_spans_only['rougeL'], results_llm_spans_only['recall'],
        results_llm_spans_only['rougeLp'], results_llm_spans_only['avg_len'], results_llm_spans_only['unans_acc']),
    ("VerbatimRAG-Semantic (contextual template)",
        results_semantic['rougeL'], results_semantic['recall'],
        results_semantic['rougeLp'], results_semantic['avg_len'], results_semantic['unans_acc']),
    ("VerbatimRAG-Semantic (spans only)",
        results_semantic_spans_only['rougeL'], results_semantic_spans_only['recall'],
        results_semantic_spans_only['rougeLp'], results_semantic_spans_only['avg_len'], results_semantic_spans_only['unans_acc']),
]

headers = ["Model", "RougeL", "Recall", "RougeLp", "Len", "Unans\\%"]
all_rows = paper_baselines + our_results
n_paper = len(paper_baselines)

# LaTeX
latex = tabulate(all_rows, headers=headers, tablefmt="latex_booktabs", floatfmt=".1f")
lines = latex.split("\n")
data_count, result = 0, []
for line in lines:
    result.append(line)
    if "&" in line:
        data_count += 1
        if data_count == n_paper:
            result.append(r"\midrule")
print("\n".join(result))

# Markdown
print("\n\n--- Markdown ---")
print(tabulate(all_rows, headers=headers, tablefmt="pipe", floatfmt=".1f"))

\begin{tabular}{lrrrrr}
\toprule
 Model                                      &   RougeL &   Recall &   RougeLp &   Len &   Unans\textbackslash{}\% \\
\midrule
 FLAN-T5-XXL 1/0                            &     31.9 &     23.6 &      15.0 &  75.0 &      78.1 \\
 Llama-13B-chat                             &     35.5 &     64.3 &      34.0 & 491.0 &      25.0 \\
 Mistral-7B-Instruct                        &     39.0 &     56.0 &      29.0 & 384.0 &      18.6 \\
 GPT-3.5                                    &     39.8 &     58.9 &      30.0 & 444.0 &      37.0 \\
 GPT-4                                      &     35.9 &     67.7 &      30.0 & 759.0 &      18.0 \\
 CLAPNQ-T5-LG-200                           &     41.5 &     51.3 &      42.1 & 272.0 &      89.7 \\
 CLAPNQ-T5-LG                               &     57.2 &     68.3 &      51.0 & 318.0 &      89.2 \\
\midrule
 Full Passage                               &     49.5 &     97.4 &     100.0 & 912.0 &       0.0 \\
 VerbatimRAG-LLM (contex